# MICAI 2026 · MLOps Workshop
## Notebook 1 of 3 · Data Quality, EDA and Dataset Versioning
**Chihuahua, Mexico · November 2, 2026**

| | |
|---|---|
| **Workshop** | MICAI 2026 · MLOps Workshop: from raw data to a monitored model |
| **Dataset** | *Wine Quality* (white wine). P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis (2009). *Modeling wine preferences by data mining from physicochemical properties.* Decision Support Systems 47(4), 547–553. Obtained from the UCI Machine Learning Repository. The file `modified.csv` is a deliberately corrupted copy prepared for this workshop. |
| **Tools** | Python, pandas, scikit-learn, SciPy, MLflow, Databricks and Google Colab |

**Declaration on the use of AI.** These materials were developed with the assistance of generative AI
(Anthropic's Claude, model Claude Opus 5.5), used to draft and review code, explanations and documentation.
The workshop instructors reviewed and validated all content and are responsible for it.

    ## What you will learn

    Real data is never clean. Before training any model we must answer: **can we trust this data?**
    In this notebook you will:

    1. Load a raw file **safely** and write down a **data contract** (what valid data looks like).
    2. Run **diagnostic tests** that measure each data-quality problem before changing anything.
    3. Clean the data with a **pipeline of small, testable steps**, using two strategies: drop or impute.
    4. Prove with **exploratory data analysis (EDA)** and statistical tests that the cleaned data tells the same story
       as the original.
    5. **Version** the cleaned datasets with **MLflow**, so every model can be traced back to the exact data it used.

    **The workshop at a glance**

    | # | Notebook | Question it answers | MLflow experiment |
|---|---|---|---|
| 0 | Set up your own workspace (once, on your own Databricks) | Is everything ready? | `00_official_datasets` |
| 1 | **➡ Data quality, EDA and versioning** | Can we trust the data? | `01_data_cleaning` |
| 2 | Training and experiment tracking | Which model, trained on which data version, works best? | `02_training` |
| 3 | Monitoring and data drift | Is the model still valid once the data changes? | `03_drift_monitoring` |

    **How to use this notebook**
    - Run the cells from top to bottom: click a cell and press **Shift + Enter** (or use the ▶ button).
    - Every code cell is preceded by a text cell that explains what it does and why.
    - Sections marked **Deep dive (optional)** at the end are for later; they are not needed for a first run.

## Step 0 · Setup (about 1 minute)

Run the next cell. It has two fields:

1. **CONFIG_LINK**: where the config comes from (the Databricks address and your access key).
   - **On your own Databricks:** leave it **empty**. The cell reads your Colab Secret `MICAI_CONFIG`
     (key icon 🔑 on the left; switch on *Notebook access*). How to create it:
     [setup guide](https://github.com/ITESM-MNA/MICAI_MLOps_2026/blob/main/docs/SETUP_YOUR_OWN_MLFLOW.md).
   - **In a workshop:** paste the config link given by the instructor.
   - A file path also works, for example `/content/config.json` after uploading the file.
2. **NICKNAME**: any nickname used to tag your runs. Please don't use your real name.

The cell then, automatically:
- downloads the workshop code and data from GitHub into this Colab machine,
- installs MLflow,
- reads the config (the access key is never printed),
- connects this notebook to the MLflow experiment `01_data_cleaning` on Databricks.

If anything fails (no config, a mistyped link, the network, an expired key), the cell says so in one line and
switches to a **local** MLflow store inside Colab. The notebook still runs end to end; your runs are just not saved
on Databricks.

> Each notebook runs on its own Colab machine, so this setup is repeated at the top of every notebook,
> with the same config and nickname.

In [ ]:
#@title Step 0 · Workshop setup (click ▶ to run)
CONFIG_LINK = ""  #@param {type:"string"}
NICKNAME = ""  #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ITESM-MNA/MICAI_MLOps_2026.git"
try:
    import google.colab  # noqa: F401  (only available inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = Path("/content/MICAI_MLOps_2026")
    if not REPO_DIR.exists():
        print("Downloading the workshop files...")
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, str(REPO_DIR)], check=True)
    print("Installing MLflow (about 20 seconds)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements-colab.txt")], check=True)
else:  # running outside Colab, from the notebooks/ folder of the repository
    here = Path.cwd()
    REPO_DIR = Path(os.environ.get("MICAI_REPO_DIR", here.parent if (here.parent / "src").exists() else here))

sys.path.insert(0, str(REPO_DIR / "src"))
from micai_mlops.setup import setup_workshop
ws = setup_workshop(CONFIG_LINK, NICKNAME, notebook="01", repo_dir=REPO_DIR)

## 1. Load the raw data safely

The file `modified.csv` is a copy of the UCI *Wine Quality* (white wine) dataset that was **deliberately corrupted**
to look like a real raw export: padded spaces, placeholder words such as `"?"` or `"error"`, impossible values,
a noise column and duplicated rows. Each row is one wine: 11 lab measurements plus a sensory `quality` score (0–10).

**Why read everything as text?** If pandas guesses the types, a single `"?"` silently turns a whole numeric column
into text, and its default list of missing-value markers hides problems we want to *see*.
`dtype=str` and `keep_default_na=False` give us the file exactly as it is; we convert types explicitly later.
This is the MLOps principle **"never trust the input, validate it"**.

**Why add a `row_id`?** It gives every raw record a stable identifier, so any cleaned row can be traced back to the
line of the raw file it came from (*data lineage*). It is never used as a model feature.

In [ ]:
import json
import tempfile

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA_DIR = ws.data_dir
raw_df = pd.read_csv(DATA_DIR / "modified.csv", dtype=str, keep_default_na=False)
raw_df.insert(0, "row_id", np.arange(len(raw_df), dtype="int64"))

print(f"Rows: {raw_df.shape[0]:,}   Columns: {raw_df.shape[1]}")
raw_df.head()

## 2. Write down the data contract

A **data contract** is the written agreement of what valid data looks like: the expected columns, their types and
their plausible ranges. Tools such as *pandera* or *Great Expectations* formalize it; here it is a plain dictionary so
every rule is visible.

The bounds come from oenology and lab-measurement knowledge, **not** from the data. They are deliberately generous:
they catch **impossible** values (a pH of 700), not unusual but real wines.

In [ ]:
# column: (min, max), inclusive. Units in the comments.
DATA_CONTRACT = {
    "fixed acidity":        (3.0,   16.0),   # g(tartaric acid)/L
    "volatile acidity":     (0.05,  1.6),    # g(acetic acid)/L; above ~1.2 wine is vinegar-like
    "citric acid":          (0.0,   1.7),    # g/L, cannot be negative
    "residual sugar":       (0.0,   70.0),   # g/L
    "chlorides":            (0.005, 0.7),    # g(sodium chloride)/L
    "free sulfur dioxide":  (1.0,   300.0),  # mg/L
    "total sulfur dioxide": (5.0,   450.0),  # mg/L
    "density":              (0.985, 1.04),   # g/cm^3, wine is close to water
    "pH":                   (2.7,   4.0),    # wine is acidic
    "sulphates":            (0.2,   2.0),    # g(potassium sulphate)/L
    "alcohol":              (8.0,   15.5),   # % vol for still table wine
    "quality":              (0,     10),     # sensory score, integer 0-10 (the target)
}
ROW_ID = "row_id"
TARGET = "quality"
EXPECTED_COLUMNS = list(DATA_CONTRACT)
FEATURES = [col for col in EXPECTED_COLUMNS if col != TARGET]
print(f"{len(FEATURES)} features + target '{TARGET}' + lineage column '{ROW_ID}'")

## 3. Diagnostic tests: measure before you clean

Before changing anything we **measure**. Each test answers one question and points to one cleaning action:

| # | Test | If it fails → action |
|---|---|---|
| T1 | Schema: do the columns match the contract? | Drop unexpected columns; stop if a required one is missing |
| T2 | Type: can every value be parsed as a number? | Cast types explicitly, after the fixes below |
| T3 | Whitespace: values with leading/trailing spaces? | Strip spaces |
| T4 | Placeholders: which non-numeric words appear? | Map them to missing |
| T5 | Missing values: how many empty cells per column? | Decide: drop or impute |
| T6 | Domain range: values outside the contract bounds? | Treat as invalid (missing) |
| T7 | Cross-field: is free SO₂ ≤ total SO₂? | Treat both values as invalid |
| T8 | Target: is `quality` an integer in 0–10? | Drop the row: **never impute a label** |
| T9 | Duplicates: exact duplicate rows? | Drop them, **after** normalization |

### T1 · Schema test
An extra column can leak noise (or even the answer) into a model; a missing column breaks the pipeline.

In [ ]:
extra_cols = [col for col in raw_df.columns if col not in EXPECTED_COLUMNS + [ROW_ID]]
missing_cols = [col for col in EXPECTED_COLUMNS if col not in raw_df.columns]
print("Unexpected columns:", extra_cols)
print("Missing columns:   ", missing_cols)

# Inspect the unexpected column before deciding what to do with it
for col in extra_cols:
    as_number = pd.to_numeric(raw_df[col].str.strip(), errors="coerce")
    quality = pd.to_numeric(raw_df[TARGET].str.strip(), errors="coerce")
    print(f"\n'{col}': {raw_df[col].nunique():,} distinct values, "
          f"{as_number.notna().mean():.0%} parseable as numbers")
    print("Most frequent values:", raw_df[col].value_counts().head(5).to_dict())
    print(f"Spearman correlation with quality: {as_number.corr(quality, method='spearman'):.3f}")

**Result → action.** `mixed_type_col` is not in the contract, mixes numbers with words (`unknown`, `bad`) and has no
relationship with `quality`. It is a noise column → **drop it**.

### T2–T5 · Types, whitespace, placeholders and missing values
`" 0.27 "` and `"0.27"` are the same measurement but different strings: padding breaks parsing and hides duplicates.
Upstream systems also write placeholders (`?`, `n/a`, `error`…) instead of leaving a cell empty. We list them
explicitly rather than guess, so a *new* unexpected word is noticed the next time the pipeline runs.

In [ ]:
text = raw_df[EXPECTED_COLUMNS]
stripped = text.apply(lambda s: s.str.strip())

def parse_rate(series):
    return pd.to_numeric(series, errors="coerce").notna().mean()

# T4: words that are not numbers, after stripping spaces
not_numeric = stripped.apply(lambda s: s[(s != "") & pd.to_numeric(s, errors="coerce").isna()])
token_counts = pd.Series(not_numeric.stack().values).value_counts()
PLACEHOLDER_TOKENS = {token.lower() for token in token_counts.index}
print("Placeholder words found:", token_counts.to_dict())
print("Treated as missing (case-insensitive):", sorted(PLACEHOLDER_TOKENS))

per_column = pd.DataFrame({
    "T2 % parseable as-is": (100 * text.apply(parse_rate)).round(1),
    "T3 cells with spaces": (text != stripped).sum(),
    "T4 placeholder words": stripped.apply(lambda s: s.str.lower().isin(PLACEHOLDER_TOKENS)).sum(),
    "T5 empty cells": (stripped == "").sum(),
})
per_column

**Result → action.** About 5% of the cells are padded with spaces (**strip them**), a small fixed set of
placeholder words appears in every column (**map them to missing**), and only then can we **cast to numbers**.

### T6–T9 · Ranges, consistency, target and duplicates

> **Error vs. outlier.** A domain rule flags *impossible* values (a pH of 700): those are **errors** and are removed.
> A statistical rule such as the IQR rule flags *unusual* values, many of which are real wines. Statistical rules are
> kept as diagnostics only.

In [ ]:
numeric = stripped.apply(lambda s: pd.to_numeric(s, errors="coerce"))

# T6: values outside the contract bounds (with a few examples)
range_test = pd.DataFrame([
    {"feature": col, "min allowed": lo, "max allowed": hi,
     "observed min": numeric[col].min(), "observed max": numeric[col].max(),
     "out of range": int(((numeric[col] < lo) | (numeric[col] > hi)).sum())}
    for col, (lo, hi) in DATA_CONTRACT.items()
]).set_index("feature")
display(range_test)

# T7: free SO2 is part of total SO2, so free <= total must hold
so2_violations = int((numeric["free sulfur dioxide"] > numeric["total sulfur dioxide"]).sum())

# T8: the target must be an integer between 0 and 10
q = numeric[TARGET]
bad_target = int(q.isna().sum() + ((q < 0) | (q > 10)).sum() + (q.dropna() % 1 != 0).sum())

# T9: duplicates hidden by formatting noise
print(f"T7 rows with free SO2 > total SO2:       {so2_violations}")
print(f"T8 rows with a missing/invalid target:   {bad_target}")
print(f"T9 exact duplicates in the raw file:     {raw_df.drop(columns=ROW_ID).duplicated().sum()}")
print(f"T9 ... ignoring the noise column:        {text.duplicated().sum()}")
print(f"T9 ... after stripping and parsing:      {numeric.duplicated().sum()}")

**Result → action.**
- **T6:** every column has impossible values, often 10–1000× the normal scale → set them to missing.
- **T7:** when free SO₂ > total SO₂ we cannot know which value is wrong → set **both** to missing.
- **T8:** rows with a missing or impossible `quality` → drop them. A guessed label is a fabricated example.
- **T9:** the duplicate count grows after each normalization step, so formatting noise was *hiding* duplicates →
  remove duplicates as the **last** step.

## 4. The cleaning pipeline

Each test maps to one small **pure function**: a DataFrame goes in, a new DataFrame comes out, with no hidden state.
This is how cleaning code is written in MLOps: every step can be tested on its own, the order is explicit, and the
same pipeline can run unchanged on next month's data or inside a production job.

A **cleaning log** records what each step did. We will store it in MLflow together with the cleaned data.

In [ ]:
cleaning_log = []

def log_step(step, detail, rows_before, rows_after, cells_changed=None):
    cleaning_log.append({"step": step, "detail": detail, "rows_before": rows_before,
                         "rows_after": rows_after, "cells_changed": cells_changed})

def enforce_schema(df):                       # T1
    dropped = [col for col in df.columns if col not in EXPECTED_COLUMNS + [ROW_ID]]
    missing = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")   # fail loudly, never guess
    log_step("enforce_schema", f"dropped {dropped}", len(df), len(df))
    return df[[ROW_ID] + EXPECTED_COLUMNS].copy()

def strip_whitespace(df):                     # T3
    out = df.copy()
    out[EXPECTED_COLUMNS] = df[EXPECTED_COLUMNS].apply(lambda s: s.str.strip())
    changed = int((out[EXPECTED_COLUMNS] != df[EXPECTED_COLUMNS]).sum().sum())
    log_step("strip_whitespace", "str.strip() on all columns", len(df), len(out), changed)
    return out

def placeholders_to_nan(df, tokens):          # T4, T5
    out = df.copy()
    is_placeholder = df[EXPECTED_COLUMNS].apply(lambda s: s.str.lower().isin(tokens) | (s == ""))
    out[EXPECTED_COLUMNS] = df[EXPECTED_COLUMNS].mask(is_placeholder)
    log_step("placeholders_to_nan", f"{len(tokens)} tokens + empty strings", len(df), len(out),
             int(is_placeholder.sum().sum()))
    return out

def cast_numeric(df):                         # T2
    values = df[EXPECTED_COLUMNS].apply(lambda s: pd.to_numeric(s, errors="coerce")).astype("float64")
    newly_nan = int((values.isna() & df[EXPECTED_COLUMNS].notna()).sum().sum())
    out = pd.concat([df[[ROW_ID]], values], axis=1)
    log_step("cast_numeric", "to float64 (unparseable -> NaN)", len(df), len(out), newly_nan)
    return out

def invalidate_out_of_range(df):              # T6
    out = df.copy()
    n = 0
    for col, (lo, hi) in DATA_CONTRACT.items():
        bad = (out[col] < lo) | (out[col] > hi)
        n += int(bad.sum())
        out.loc[bad, col] = np.nan
    log_step("invalidate_out_of_range", "values outside data-contract bounds -> NaN", len(df), len(out), n)
    return out

def invalidate_inconsistent_so2(df):          # T7
    out = df.copy()
    bad = out["free sulfur dioxide"] > out["total sulfur dioxide"]
    out.loc[bad, ["free sulfur dioxide", "total sulfur dioxide"]] = np.nan
    log_step("invalidate_inconsistent_so2", "free SO2 > total SO2 -> both NaN", len(df), len(out),
             int(bad.sum()) * 2)
    return out

def drop_invalid_target(df):                  # T8
    valid = df[TARGET].notna() & (df[TARGET] % 1 == 0)
    out = df[valid].copy()
    out[TARGET] = out[TARGET].astype("int64")
    log_step("drop_invalid_target", "rows with missing/invalid quality removed", len(df), len(out))
    return out

def drop_duplicates(df):                      # T9 (judged on the values, never on row_id)
    out = df.drop_duplicates(subset=EXPECTED_COLUMNS).copy()
    log_step("drop_duplicates", "exact duplicate rows removed", len(df), len(out))
    return out

### Steps 1–7, shared by both strategies
`.pipe()` chains the steps in an explicit order. At the end every problem has been turned into a single, well
understood one: **missing feature values**.

In [ ]:
base_df = (
    raw_df
    .pipe(enforce_schema)
    .pipe(strip_whitespace)
    .pipe(placeholders_to_nan, PLACEHOLDER_TOKENS)
    .pipe(cast_numeric)
    .pipe(invalidate_out_of_range)
    .pipe(invalidate_inconsistent_so2)
    .pipe(drop_invalid_target)
)
rows_with_nan = base_df[FEATURES].isna().any(axis=1).sum()
print(f"Rows after the shared steps: {len(base_df):,}")
print(f"Rows with at least one missing feature: {rows_with_nan:,} ({rows_with_nan / len(base_df):.1%})")

## 5. Two strategies for the missing values

| | **Strategy A: drop incomplete rows** | **Strategy B: impute with the median** |
|---|---|---|
| Idea | Keep only rows where every feature is valid | Replace each missing value with the column median |
| Pros | Every value is a real measurement; distributions are not distorted | Keeps many more rows |
| Cons | Loses every row with *any* bad cell (here about a quarter) | Creates artificial spikes at the median, shrinks the spread |
| Valid when | Missingness is random and enough rows remain | Missingness is low and every row is needed |

> **MLOps note on leakage.** In a modelling pipeline the median must be computed on the training data only and then
> applied to the test data. Here we impute the whole file because the goal is cleaning for EDA; notebook 2 evaluates
> every model on a fixed holdout of real measurements to keep the comparison fair.

Imputation works column by column, so it can break a multi-column rule: an imputed free SO₂ may exceed a real total
SO₂. Strategy B re-checks T7 after imputing. Deduplication (step 9) runs last in both strategies.

In [ ]:
def strategy_a_drop_rows(df):
    out = df.dropna(subset=FEATURES).copy()
    log_step("A: drop_incomplete_rows", "rows with any missing feature removed", len(df), len(out))
    return out

def strategy_b_impute_median(df):
    out = df.copy()
    medians = out[FEATURES].median()
    n = int(out[FEATURES].isna().sum().sum())
    out[FEATURES] = out[FEATURES].fillna(medians)
    log_step("B: impute_median", "missing features <- column median", len(df), len(out), n)
    consistent = out["free sulfur dioxide"] <= out["total sulfur dioxide"]
    log_step("B: recheck_so2_after_impute", "rows made inconsistent by imputation removed",
             len(out), int(consistent.sum()))
    return out[consistent].copy()

clean_a = base_df.pipe(strategy_a_drop_rows).pipe(drop_duplicates).reset_index(drop=True)
clean_b = base_df.pipe(strategy_b_impute_median).pipe(drop_duplicates).reset_index(drop=True)

print(f"Strategy A (drop rows): {len(clean_a):,} rows")
print(f"Strategy B (impute):    {len(clean_b):,} rows")
pd.DataFrame(cleaning_log)

## 6. Validation gate

Cleaning is not finished until we **prove** that the output satisfies the data contract. In a production pipeline
these assertions run automatically before training; if one fails, the pipeline stops.

In [ ]:
def validate(df, name):
    assert list(df.columns) == [ROW_ID] + EXPECTED_COLUMNS, "column set and order"
    assert df[ROW_ID].is_unique, "row_id is unique"
    assert df.notna().all().all(), "no missing values"
    assert pd.api.types.is_integer_dtype(df[TARGET]), "integer target"
    for col, (lo, hi) in DATA_CONTRACT.items():
        assert df[col].between(lo, hi).all(), f"{col} within contract bounds"
    assert (df["free sulfur dioxide"] <= df["total sulfur dioxide"]).all(), "SO2 consistency"
    assert not df.duplicated(subset=EXPECTED_COLUMNS).any(), "no duplicate rows"
    print(f"✅ {name}: all {len(df):,} rows pass the data contract")

validate(clean_a, "Clean A (drop rows)")
validate(clean_b, "Clean B (impute)")

## 7. EDA: does the cleaned data tell the same story?

We now explore the data and, at the same time, check that cleaning did not distort it. Because this is a teaching
exercise we have the uncorrupted file `original.csv`; in a real project there is no such file. It is used **only
here**, as a baseline, and never to make cleaning decisions.

| Name | What it is |
|---|---|
| **Original (dedup)** | Baseline: `original.csv` with its duplicate rows removed by the same rule (about 19% of the original file are repeats) |
| **Clean A – drop** | Output of strategy A |
| **Clean B – impute** | Output of strategy B |

### 7.1 The target: `quality`
The class balance matters for modelling: if it changed during cleaning, every later model would be biased.
In notebook 2 we predict whether a wine is **good (quality ≥ 7)**, so its share is shown too.

In [ ]:
original_dedup = pd.read_csv(DATA_DIR / "original.csv").drop_duplicates().reset_index(drop=True)
datasets = {
    "Original (dedup)": original_dedup,
    "Clean A - drop": clean_a[EXPECTED_COLUMNS],
    "Clean B - impute": clean_b[EXPECTED_COLUMNS],
}
REFERENCE = "Original (dedup)"
PALETTE = dict(zip(datasets, ["#4c72b0", "#55a868", "#dd8452"]))

quality_pct = pd.DataFrame({name: d[TARGET].value_counts(normalize=True).sort_index() * 100
                            for name, d in datasets.items()}).fillna(0)
good_share = pd.Series({name: (d[TARGET] >= 7).mean() * 100 for name, d in datasets.items()},
                       name="% good (quality >= 7)")

fig, ax = plt.subplots(figsize=(10, 4))
quality_pct.plot.bar(ax=ax, color=list(PALETTE.values()), rot=0)
ax.set(title="Distribution of wine quality (% of wines)", xlabel="quality", ylabel="% of wines")
plt.show()
display(good_share.round(1).to_frame())

ref_counts = datasets[REFERENCE][TARGET].value_counts().sort_index()
for name in ["Clean A - drop", "Clean B - impute"]:
    counts = datasets[name][TARGET].value_counts().reindex(ref_counts.index, fill_value=0)
    chi2, p, _, _ = stats.chi2_contingency(np.vstack([ref_counts.values, counts.values]))
    print(f"Chi-square test {name} vs {REFERENCE}: p-value = {p:.3f}")

**How to read it.** Qualities 5–7 dominate and 3 and 9 are rare: the data is **imbalanced**, and only about one
wine in five is *good*. A chi-square p-value above 0.05 means no evidence that cleaning changed the class proportions,
because rows were removed at random positions, never based on the label.

### 7.2 Feature distributions
Each panel overlays the three datasets, normalized to the same area.

In [ ]:
long_df = pd.concat([d[FEATURES].assign(dataset=name) for name, d in datasets.items()], ignore_index=True)
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
for ax, col in zip(axes.flat, FEATURES):
    sns.kdeplot(data=long_df, x=col, hue="dataset", palette=PALETTE, common_norm=False, ax=ax,
                linewidth=1.5, legend=(col == FEATURES[0]))
    ax.set_title(col)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

**How to read it.** The cleaned curves lie on top of the original. For Strategy B, look for a taller, sharper peak in
some variables (for example `alcohol`, `pH`, `sulphates`): the visual signature of median imputation, where 1–2% of
each column was set to the same value.

### 7.3 Potential outliers (IQR rule, as a diagnostic)
The share of values outside 1.5 × IQR. Similar numbers across the three datasets mean cleaning removed the injected
**errors** but kept the natural tails: unusual real wines are still there, as they should be.

In [ ]:
def iqr_outlier_pct(d):
    result = {}
    for col in FEATURES:
        q1, q3 = d[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        result[col] = 100 * ((d[col] < q1 - 1.5 * iqr) | (d[col] > q3 + 1.5 * iqr)).mean()
    return pd.Series(result)

pd.DataFrame({name: iqr_outlier_pct(d) for name, d in datasets.items()}).round(2)

### 7.4 Relationships with quality, and among predictors
**Pearson** correlation measures *linear* relationships; **Spearman** works on ranks and also captures monotonic,
non-linear ones. The ranking of features by their correlation with `quality` often drives feature selection, so it
must not change because of cleaning.

In [ ]:
corr_with_quality = pd.DataFrame({
    f"{name} · {method}": d[FEATURES + [TARGET]].corr(method=method)[TARGET].drop(TARGET)
    for name, d in datasets.items() for method in ["pearson", "spearman"]
})
display(corr_with_quality.sort_values(f"{REFERENCE} · pearson", key=abs, ascending=False).round(3))

# Strongly correlated predictor pairs (Clean A): redundant information for a model
corr = clean_a[FEATURES].corr()
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
         .rename("pearson").reset_index().rename(columns={"level_0": "feature 1", "level_1": "feature 2"}))
pairs.reindex(pairs["pearson"].abs().sort_values(ascending=False).index).head(6).round(3)

**How to read it.** `alcohol` (positive) and `density` (negative) are the features most related to quality in every
dataset. `density` is strongly tied to `residual sugar` and `alcohol`: physically, sugar makes wine denser and alcohol
makes it lighter. Remember `density`: it comes back in notebook 3.

### 7.5 Scorecard: formal distribution tests
Plots are persuasive but subjective. Three numbers make "the distributions match" measurable:

| Metric | What it measures | "Equivalent" when |
|---|---|---|
| Two-sample **Kolmogorov–Smirnov (KS)** | Largest gap between the two cumulative distributions | p-value > 0.05 |
| **Standardized mean difference (SMD)** | Shift of the mean, in reference standard deviations | \|SMD\| < 0.1 |
| **Variance ratio** | Spread of cleaned / spread of reference | between 0.9 and 1.1 |

In [ ]:
def distribution_tests(name):
    ref, d = datasets[REFERENCE], datasets[name]
    rows = []
    for col in FEATURES:
        rows.append({"feature": col,
                     "ks_pvalue": stats.ks_2samp(ref[col], d[col]).pvalue,
                     "smd": (d[col].mean() - ref[col].mean()) / ref[col].std(),
                     "variance_ratio": d[col].var() / ref[col].var()})
    out = pd.DataFrame(rows).set_index("feature")
    out["equivalent"] = (out["ks_pvalue"] > 0.05) & (out["smd"].abs() < 0.1) & out["variance_ratio"].between(0.9, 1.1)
    return out

scorecard = []
for name in ["Clean A - drop", "Clean B - impute"]:
    t = distribution_tests(name)
    corr_gap = (datasets[name][FEATURES + [TARGET]].corr()
                - datasets[REFERENCE][FEATURES + [TARGET]].corr()).abs().max().max()
    scorecard.append({"cleaned dataset": name, "rows": len(datasets[name]),
                      "features equivalent": f"{int(t['equivalent'].sum())}/{len(t)}",
                      "not equivalent": ", ".join(t.index[~t["equivalent"]]) or "-",
                      "min KS p-value": round(t["ks_pvalue"].min(), 3),
                      "max |SMD|": round(t["smd"].abs().max(), 3),
                      "max |Δ correlation|": round(corr_gap, 3)})
pd.DataFrame(scorecard).set_index("cleaned dataset")

**Conclusion.**
- **Strategy A (drop rows)** keeps only real measurements, so its distributions and correlations match the original
  closely. It costs about a quarter of the rows, which is affordable here because the corruption was random and
  thousands of rows remain.
- **Strategy B (impute)** keeps more rows, but the median spikes are detectable by the KS test in a couple of
  features. Class balance and the ranking of important features are preserved, so B remains usable when rows are
  scarce, as long as its bias is acknowledged.

Which version produces the **better model**? Data quality metrics cannot answer that; **notebook 2 trains on both
and lets the results decide.** For that, both versions must be stored and traceable.

## 8. Version the datasets with MLflow

A model is only reproducible if we know **exactly which data** it was trained on. MLflow records datasets next to
experiments:

- `mlflow.data.from_pandas(...)` describes a dataset and computes a **digest**: a short fingerprint of its content.
- `mlflow.log_input(...)` attaches it to a run, with a **context** label (raw, cleaned, training…). In the MLflow
  UI it appears in the run's **Datasets** column.
- `mlflow.log_artifacts(...)` stores the files themselves, so they can be downloaded later.

We also log the **data contract** and the **cleaning log**, so anyone can see *how* each version was produced.

The next cell creates one run in the experiment `01_data_cleaning`, tagged with your nickname.

> **One official copy.** Notebooks 2 and 3 do **not** read the versions you log here. They use the **official**
> versions stored in the experiment `00_official_datasets`, produced by exactly the same code: published by
> notebook 0 when you run on your own Databricks, or by the instructor in a workshop, where many people clean at
> the same time. After logging, we check that your versions are identical to the official ones.

In [ ]:
from micai_mlops.fingerprint import canonical_csv, fingerprint

versions = {"v1_clean_drop": clean_a, "v2_clean_impute": clean_b}

with mlflow.start_run(run_name=f"{ws.alias}-cleaning") as run:
    mlflow.set_tags({"alias": ws.alias, "notebook": "01"})
    mlflow.log_params({"placeholder_tokens": ",".join(sorted(PLACEHOLDER_TOKENS)),
                       "strategy_a": "drop incomplete rows", "strategy_b": "median imputation",
                       "deduplication": "exact duplicates, after normalization"})
    # v0: the raw file exactly as received
    mlflow.log_input(mlflow.data.from_pandas(raw_df, name="v0_raw", source="data/wine-quality/modified.csv"),
                     context="raw")
    # v1 and v2: the cleaned versions
    for name, df in versions.items():
        dataset = mlflow.data.from_pandas(df, name=name, targets=TARGET)
        mlflow.log_input(dataset, context="cleaned")
        mlflow.log_metric(f"rows_{name}", len(df))
        print(f"{name:16} rows={len(df):,}  MLflow digest={dataset.digest}")
    with tempfile.TemporaryDirectory() as tmp:
        for name, df in versions.items():
            with open(f"{tmp}/{name}.csv", "w") as f:
                f.write(canonical_csv(df))
        mlflow.log_artifacts(tmp, artifact_path="datasets")
    mlflow.log_dict({col: list(bounds) for col, bounds in DATA_CONTRACT.items()}, "data_contract.json")
    mlflow.log_table(pd.DataFrame(cleaning_log), "cleaning_log.json")

print(f"\n✅ Logged run '{run.info.run_name}' in {ws.experiment('cleaning')}")
if ws.online:
    print("Open it in the MLflow UI:", ws.run_url("cleaning", run.info.run_id))

### Is your version identical to the official one?
A **fingerprint** (SHA-256 hash of the file content) identifies a dataset exactly: change one value and the
fingerprint changes. The repository contains the fingerprints of the official versions. Because the cleaning code is
deterministic, your versions should match. If you changed a rule above, they won't, and that is the point:
**a different rule is a different dataset version.**

In [ ]:
official_manifest = json.loads((DATA_DIR / "official_manifest.json").read_text())
pd.DataFrame([
    {"version": name, "your rows": len(df), "official rows": official_manifest[name]["rows"],
     "your fingerprint": fingerprint(df)[:12],
     "official fingerprint": official_manifest[name]["sha256"][:12],
     "identical": "✅" if fingerprint(df) == official_manifest[name]["sha256"] else "❌ (different version)"}
    for name, df in versions.items()
]).set_index("version")

## Summary

1. **Test first, then clean.** Every action was justified by a measured test result.
2. **Write the data contract down** and validate the output against it automatically.
3. **Errors are not outliers.** Domain rules remove impossible values; statistical rules are diagnostics.
4. **Never impute the target**, and fit any imputation on training data only.
5. **Version the data with its lineage.** Each version is logged in MLflow with its fingerprint, the contract and
   the cleaning log, so any model can be traced back to the exact data it used.

**Next: notebook 2**, where we train models on both versions and compare them in MLflow.

---
## Deep dive (optional): audit the cleaning against the original
Because the corruption was synthetic, we can check cell by cell how many corrupted values the tests caught. The first
4,898 rows of `modified.csv` are aligned with `original.csv`; the injected duplicates were appended at the end.

In [ ]:
original_df = pd.read_csv(DATA_DIR / "original.csv")
saved_log = list(cleaning_log)            # the audit re-uses the steps; keep the real log intact
parsed = (raw_df.pipe(enforce_schema).pipe(strip_whitespace)
          .pipe(placeholders_to_nan, PLACEHOLDER_TOKENS).pipe(cast_numeric))
flagged = (parsed.pipe(invalidate_out_of_range).pipe(invalidate_inconsistent_so2)
           [EXPECTED_COLUMNS].iloc[:len(original_df)].isna().to_numpy())
cleaning_log[:] = saved_log

aligned = parsed[EXPECTED_COLUMNS].iloc[:len(original_df)].to_numpy()
corrupted = ~np.isclose(aligned, original_df[EXPECTED_COLUMNS].to_numpy(), equal_nan=False)
tp, fn, fp = (flagged & corrupted).sum(), (~flagged & corrupted).sum(), (flagged & ~corrupted).sum()
print(f"Corrupted cells: {corrupted.sum():,}")
print(f"Detected by the tests: {tp:,} (recall {tp / (tp + fn):.1%})")
print(f"Valid cells wrongly flagged: {fp:,}   Undetected: {fn:,}")

The tests catch about 99% of the corrupted cells. The few "false positives" come from the SO₂ rule (T7), which
discards both values of an inconsistent pair because it cannot know which one is wrong. The misses are values that
were altered but stayed inside a plausible range: no single-row rule can detect those, and they behave like ordinary
measurement noise.